In [ ]:

import mlflow
from mlflow.tracking import MlflowClient
import dagshub

dagshub.init(repo_owner='hwilco', repo_name='fof8-scout', mlflow=True)
client = MlflowClient()

# Fetch the specific run
run_id = 'defc9cb263184352ae44a5906cfd477e'
run = client.get_run(run_id)
print('Run name:', run.info.run_name)
print('Status:', run.info.status)
print('Tags:', run.data.tags)
print()
print('Metrics:', run.data.metrics)
print()
print('Params (subset):', {k: v for k, v in run.data.params.items() if 'stage1' in k.lower() or 'threshold' in k.lower() or 'target' in k.lower()})

In [ ]:
import mlflow
import dagshub
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import train_test_split

# Initialize DagsHub/MLflow
dagshub.init(repo_owner="hwilco", repo_name="fof8-scout", mlflow=True)
client = mlflow.tracking.MlflowClient()

RUN_ID = "defc9cb263184352ae44a5906cfd477e"

# Download the OOF results artifact
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, 
    artifact_path="stage1_oof_results.csv"
)

# Load data
df_oof = pl.read_csv(artifact_path)
y_true = df_oof["y_true"].to_numpy()
y_prob = df_oof["oof_prob"].to_numpy()

print(f"Loaded {len(df_oof)} OOF predictions.")
print(f"Global Base Rate: {y_true.mean():.2%}")


In [ ]:
def plot_calibration_curve(y_true, y_prob, name, n_bins=10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    
    plt.figure(figsize=(10, 6))
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(prob_pred, prob_true, "s-", label=f"{name}")
    
    plt.ylabel("Fraction of Positives (Actual)")
    plt.xlabel("Mean Predicted Probability")
    plt.ylim([-0.05, 1.05])
    plt.legend(loc="lower right")
    plt.title(f'Reliability Curve: {name}')
    plt.grid(True, alpha=0.3)
    plt.show()

plot_calibration_curve(y_true, y_prob, "Original XGBoost/CatBoost")

# Calculate baseline metrics
print(f"Baseline Brier Score: {brier_score_loss(y_true, y_prob):.4f}")
print(f"Baseline Log Loss: {log_loss(y_true, y_prob):.4f}")


In [ ]:
# Split OOF data to validate calibration
X_calib, X_test, y_calib, y_test = train_test_split(
    y_prob.reshape(-1, 1), y_true, test_size=0.5, random_state=42
)

# 1. Platt Scaling (Logistic Regression on the outputs)
platt = LogisticRegression(C=1e10, solver='lbfgs')
platt.fit(X_calib, y_calib)
y_prob_platt = platt.predict_proba(X_test.reshape(-1, 1))[:, 1]

# 2. Isotonic Regression (Non-parametric, better if you have enough data)
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(X_calib.flatten(), y_calib)
y_prob_iso = iso.predict(X_test.flatten())

# Compare Results
methods = [
    ("Uncalibrated", X_test.flatten()),
    ("Platt Scaling", y_prob_platt),
    ("Isotonic", y_prob_iso)
]

plt.figure(figsize=(12, 8))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

for name, probs in methods:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
    score = brier_score_loss(y_test, probs)
    plt.plot(prob_pred, prob_true, "s-", label=f"{name} (Brier: {score:.4f})")

plt.ylabel("Fraction of Positives")
plt.xlabel("Mean Predicted Probability")
plt.title("Calibration Method Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Use the best method (likely Isotonic given your data size)
calibrated_probs = y_prob_iso

# Calculate what the old 0.27 threshold maps to now
# We find the samples near 0.27 in original space and see their calibrated value
idx_near_thresh = np.abs(X_test.flatten() - 0.27).argmin()
new_equivalent_thresh = calibrated_probs[idx_near_thresh]

print(f"Original Threshold: 0.27")
print(f"New Equivalent Threshold: {new_equivalent_thresh:.4f}")

# Histogram of shifts
plt.figure(figsize=(10, 5))
plt.hist(X_test.flatten(), bins=50, alpha=0.5, label='Original Probs')
plt.hist(calibrated_probs, bins=50, alpha=0.5, label='Calibrated Probs')
plt.title("Probability Distribution Shift")
plt.legend()
plt.show()


In [ ]:
# Create a range of possible raw scores
raw_range = np.linspace(0, 1, 100)

# Get transformations for both methods
platt_range = platt.predict_proba(raw_range.reshape(-1, 1))[:, 1]
iso_range = iso.predict(raw_range)

plt.figure(figsize=(10, 6))
plt.plot(raw_range, raw_range, 'k--', label='No Change (Identity)')
plt.plot(raw_range, platt_range, label='Platt Scaling (Sigmoid)')
plt.plot(raw_range, iso_range, label='Isotonic Regression (Non-parametric)')

plt.axvline(0.27, color='red', linestyle=':', label='Old Threshold (0.27)')
plt.axhline(0.1129, color='green', linestyle=':', label='New Equivalent (0.1129)')

plt.title("How Calibration Adjusts Your Raw Scores")
plt.xlabel("Raw Model Probability")
plt.ylabel("Calibrated (Real-World) Probability")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Check Brier Score Improvement
print(f"Original Brier Score: {brier_score_loss(y_test, X_test.flatten()):.4f}")
print(f"Isotonic Brier Score: {brier_score_loss(y_test, y_prob_iso):.4f}")


In [ ]:
from sklearn.linear_model import LogisticRegression

def fit_beta_calibration(y_prob, y_true):
    # To avoid log(0) or log(1) errors, we clip the probabilities slightly
    eps = 1e-10
    p = np.clip(y_prob, eps, 1 - eps)
    
    # Beta calibration uses log(p) and -log(1-p) as features
    X_beta = np.column_stack([
        np.log(p),
        -np.log(1 - p)
    ])
    
    beta_model = LogisticRegression(solver='lbfgs')
    beta_model.fit(X_beta, y_true)
    return beta_model

def predict_beta_calibration(model, y_prob):
    eps = 1e-10
    p = np.clip(y_prob, eps, 1 - eps)
    X_beta = np.column_stack([
        np.log(p),
        -np.log(1 - p)
    ])
    return model.predict_proba(X_beta)[:, 1]

# Fit and Test
beta_model = fit_beta_calibration(X_calib.flatten(), y_calib)
y_prob_beta = predict_beta_calibration(beta_model, X_test.flatten())

# Compare Metrics
beta_brier = brier_score_loss(y_test, y_prob_beta)
print(f"Isotonic Brier Score: {brier_score_loss(y_test, y_prob_iso):.4f}")
print(f"Beta Brier Score:     {beta_brier:.4f}")

# Plot Comparison
plt.figure(figsize=(10, 6))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

# Plot Isotonic (for comparison)
p_true_iso, p_pred_iso = calibration_curve(y_test, y_prob_iso, n_bins=10)
plt.plot(p_pred_iso, p_true_iso, "s-", label=f"Isotonic (Brier: {brier_score_loss(y_test, y_prob_iso):.4f})")

# Plot Beta
p_true_beta, p_pred_beta = calibration_curve(y_test, y_prob_beta, n_bins=10)
plt.plot(p_pred_beta, p_true_beta, "d-", label=f"Beta (Brier: {beta_brier:.4f})")

plt.title("Isotonic vs Beta Calibration")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
from sklearn.model_selection import StratifiedKFold

# Prepare for CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {"Uncalibrated": [], "Platt": [], "Isotonic": [], "Beta": []}

# We iterate through folds of the OOF data
for fold, (train_idx, val_idx) in enumerate(skf.split(y_prob, y_true)):
    # Training data for the calibrator
    cal_train_p, cal_val_p = y_prob[train_idx], y_prob[val_idx]
    cal_train_y, cal_val_y = y_true[train_idx], y_true[val_idx]
    
    # 1. Baseline (Uncalibrated)
    results["Uncalibrated"].append(brier_score_loss(cal_val_y, cal_val_p))
    
    # 2. Platt
    platt_cv = LogisticRegression(C=1e10).fit(cal_train_p.reshape(-1, 1), cal_train_y)
    p_platt = platt_cv.predict_proba(cal_val_p.reshape(-1, 1))[:, 1]
    results["Platt"].append(brier_score_loss(cal_val_y, p_platt))
    
    # 3. Isotonic
    iso_cv = IsotonicRegression(out_of_bounds='clip').fit(cal_train_p, cal_train_y)
    p_iso = iso_cv.predict(cal_val_p)
    results["Isotonic"].append(brier_score_loss(cal_val_y, p_iso))
    
    # 4. Beta
    beta_cv = fit_beta_calibration(cal_train_p, cal_train_y)
    p_beta = predict_beta_calibration(beta_cv, cal_val_p)
    results["Beta"].append(brier_score_loss(cal_val_y, p_beta))

# Display Results
print("--- Mean Brier Score Comparison (5-Fold CV) ---")
for method, scores in results.items():
    mean_s = np.mean(scores)
    std_s = np.std(scores)
    improvement = (results["Uncalibrated"][0] - mean_s) / results["Uncalibrated"][0]
    print(f"{method:12}: {mean_s:.5f} (+/- {std_s:.5f}) | Improvement: {improvement:.2%}")

# Quick visualization of the distribution of scores
plt.figure(figsize=(10, 5))
plt.boxplot([results[k] for k in results.keys()], labels=results.keys())
plt.ylabel("Brier Score (Lower is Better)")
plt.title("Stability of Calibration Methods Across Folds")
plt.show()


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Containers for metrics
audit_results = {
    "Original": {"alpha": [], "beta": [], "z": [], "p": []},
    "Beta": {"alpha": [], "beta": [], "z": [], "p": []}
}

for fold, (train_idx, val_idx) in enumerate(skf.split(y_prob, y_true)):
    # Data for this fold
    p_train, p_val = y_prob[train_idx], y_prob[val_idx]
    y_train, y_val = y_true[train_idx], y_true[val_idx]
    
    # 1. Calibrate (Beta)
    beta_fold_model = fit_beta_calibration(p_train, y_train)
    p_beta_val = predict_beta_calibration(beta_fold_model, p_val)
    
    # 2. Audit Original
    a_orig, b_orig = cox_calibration_audit(y_val, p_val)
    z_orig, p_orig = spiegelhalter_z_test(y_val, p_val)
    audit_results["Original"]["alpha"].append(a_orig)
    audit_results["Original"]["beta"].append(b_orig)
    audit_results["Original"]["z"].append(z_orig)
    audit_results["Original"]["p"].append(p_orig)
    
    # 3. Audit Beta
    a_beta, b_beta = cox_calibration_audit(y_val, p_beta_val)
    z_beta, p_beta = spiegelhalter_z_test(y_val, p_beta_val)
    audit_results["Beta"]["alpha"].append(a_beta)
    audit_results["Beta"]["beta"].append(b_beta)
    audit_results["Beta"]["z"].append(z_beta)
    audit_results["Beta"]["p"].append(p_beta)

print("=== 5-Fold CV Calibration Audit Summary ===")
for model in ["Original", "Beta"]:
    print(f"\n[{model} Model]")
    a_mean, b_mean = np.mean(audit_results[model]["alpha"]), np.mean(audit_results[model]["beta"])
    z_mean, p_mean = np.mean(audit_results[model]["z"]), np.mean(audit_results[model]["p"])
    
    print(f"  Cox Intercept (Alpha): {a_mean:7.4f} (Ideal: 0)")
    print(f"  Cox Slope (Beta):      {b_mean:7.4f} (Ideal: 1)")
    print(f"  Spiegelhalter Z:       {z_mean:7.4f} (Ideal: 0)")
    print(f"  Mean p-value:          {p_mean:7.4e}")
